# MiniMax H3 · Kaggle T4×2 · 视频生成
运行环境：Kaggle GPU T4 x2，Internet ON。标准 ComfyUI 单条视频目前仅使用 GPU0；GPU1 不会自动共同加速。请在执行之前确认空闲磁盘约 48GB。

In [ ]:
!nvidia-smi
!git clone -q https://github.com/mjw3177109/shipinshencheng.git /kaggle/working/shipinshencheng || true
!python /kaggle/working/shipinshencheng/kaggle/bootstrap_h3.py

## 启动 ComfyUI
若模型下载成功，启动原生服务。此 Notebook 没有公共隧道，避免无密码暴露 UI。

In [ ]:
import subprocess,os,time,requests
from pathlib import Path
comfy=Path('/kaggle/temp/minimax_h3/ComfyUI')
out=Path('/kaggle/working/h3_videos');out.mkdir(exist_ok=True)
log=open('/kaggle/working/h3_comfy.log','w')
env=os.environ.copy();env['CUDA_VISIBLE_DEVICES']='0'
proc=subprocess.Popen(['python','main.py','--listen','127.0.0.1','--port','8188','--lowvram','--disable-auto-launch','--output-directory',str(out)],cwd=comfy,env=env,stdout=log,stderr=subprocess.STDOUT)
for i in range(240):
    try:
        r=requests.get('http://127.0.0.1:8188/object_info',timeout=3)
        if r.ok: break
    except requests.RequestException: pass
    if proc.poll() is not None: raise RuntimeError('ComfyUI exited; inspect h3_comfy.log')
    time.sleep(1)
else: raise TimeoutError('ComfyUI startup timeout')
print('ComfyUI ready')

## 自动提交视频
使用 API 原生节点；如 ComfyUI 更新引起输入字段变化，API 会提供明确校验错误。

In [ ]:
import requests,uuid,json
prompt_text='电影级写实，一只金毛犬在海边栈道上，缓慢平移镜头，夕阳，海风，海浪声音，无字幕无水印。'
width,height,frames,seed=384,224,22,12345
P={
'1':{'class_type':'UNETLoader','inputs':{'unet_name':'minimax_h3_fl2va_pruned_int8_convrot.safetensors','weight_dtype':'default'}},
'2':{'class_type':'CLIPLoader','inputs':{'clip_name':'qwen3vl_32b_minimax_h3_int8_convrot.safetensors','type':'minimax','device':'default'}},
'3':{'class_type':'VAELoader','inputs':{'vae_name':'minimax_h3_video_vae_int8_convrot.safetensors'}},
'4':{'class_type':'VAELoader','inputs':{'vae_name':'minimax_h3_audio_vae_fp32.safetensors'}},
'5':{'class_type':'LoraLoaderModelOnly','inputs':{'model':['1',0],'lora_name':'minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors','strength_model':1.0}},
'6':{'class_type':'MiniMaxH3SigmaShift','inputs':{'model':['5',0],'shift_video':12.0,'shift_audio':3.0}},
'7':{'class_type':'MiniMaxH3ImageToVideo','inputs':{'clip':['2',0],'vae':['3',0],'prompt':prompt_text,'width':width,'height':height,'length':frames}},
'8':{'class_type':'RandomNoise','inputs':{'noise_seed':seed}},
'9':{'class_type':'BasicGuider','inputs':{'model':['6',0],'conditioning':['7',0]}},
'10':{'class_type':'KSamplerSelect','inputs':{'sampler_name':'euler'}},
'11':{'class_type':'BasicScheduler','inputs':{'model':['6',0],'scheduler':'simple','steps':8,'denoise':1.0}},
'12':{'class_type':'SamplerCustomAdvanced','inputs':{'noise':['8',0],'guider':['9',0],'sampler':['10',0],'sigmas':['11',0],'latent_image':['7',1]}},
'13':{'class_type':'VAEDecode','inputs':{'samples':['12',0],'vae':['3',0]}},
'14':{'class_type':'VAEDecodeAudio','inputs':{'samples':['12',0],'vae':['4',0]}},
'15':{'class_type':'CreateVideo','inputs':{'images':['13',0],'audio':['14',0],'fps':24.0,'bit_depth':8}},
'16':{'class_type':'SaveVideo','inputs':{'video':['15',0],'filename_prefix':'H3_Kaggle','format':'auto','codec':'auto'}}}
r=requests.post('http://127.0.0.1:8188/prompt',json={'prompt':P,'client_id':str(uuid.uuid4())},timeout=90)
print(r.status_code,r.text[:3000]);r.raise_for_status();prompt_id=r.json()['prompt_id'];print('task:',prompt_id)

In [ ]:
import time,json
from IPython.display import display,Video
for t in range(1440):
  history=requests.get('http://127.0.0.1:8188/history/'+prompt_id,timeout=30).json()
  if prompt_id in history:
    status=history[prompt_id].get('status',{})
    if status.get('status_str')=='error': raise RuntimeError(json.dumps(status,ensure_ascii=False)[:5000])
    break
  time.sleep(5)
else: raise TimeoutError('Render did not finish in 2 hours')
movies=sorted(out.rglob('*.mp4'),key=lambda p:p.stat().st_mtime,reverse=True)
print(movies[:3])
if movies: display(Video(str(movies[0]),embed=True,width=640))

## 限制
这是未在用户的 Kaggle 会话上验收的自动化实现。T4 显存与 CPU RAM 可能不足以跑 H3；出现 OOM 时不能认定双卡显存可相加。模型与 LoRA 的上游更新也可能改变兼容性。生成视频存于 `/kaggle/working/h3_videos`，请及时下载。